In [25]:
import torch
import numpy as np
import faiss
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

In [2]:
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DATASET_ID = "facebook/belebele"
LANGUAGE_SUBSET = "eng_Latn" 

In [3]:
def load_data():
    print(f"Loading dataset {DATASET_ID} ({LANGUAGE_SUBSET})...")
    # Load the specific language subset (English)
    ds = load_dataset(DATASET_ID, LANGUAGE_SUBSET, split="test")
    
    # Extract unique passages to simulate the FLORES knowledge base
    # Belebele is built on FLORES-200. We use the unique passages as our "Library".
    # NOTE: The column name is 'flores_passage', not 'passage'
    unique_passages = list(set(ds['flores_passage']))
    print(f"Found {len(unique_passages)} unique passages to index.")
    
    return ds, unique_passages

data = load_data()

Loading dataset facebook/belebele (eng_Latn)...
Found 488 unique passages to index.


In [ ]:
data

In [41]:
class RAGRetriever:
    def __init__(self, passages):
        self.passages = passages
        print("Loading embedding model...")
        self.encoder = SentenceTransformer(EMBEDDING_MODEL)
        
        print("Encoding passages (this may take a moment)...")
        self.embeddings = self.encoder.encode(passages, convert_to_numpy=True, show_progress_bar=True)
        
        # Initialize FAISS Index (L2 Distance)
        self.dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(self.embeddings)
        print(f"Index built with {self.index.ntotal} documents.")

    def retrieve(self, queries, k=1):
        """Retrieve top k passages for a list of queries"""
        # Encode batch
        query_vecs = self.encoder.encode(queries, convert_to_numpy=True, show_progress_bar=False)
        distances, indices = self.index.search(query_vecs, k)
        
        batch_results = []
        for i in range(len(queries)):
            # Get top k results for query i
            results = [self.passages[idx] for idx in indices[i]]
            batch_results.append(results)
        return batch_results

In [42]:
def load_llama():
    print("Loading Llama-3.1-8B (4-bit quantized)...")
    
    # quantization_config reduces memory usage to fit on consumer GPUs (Colab T4, etc.)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto"
        )
    except OSError as e:
        print("\nERROR: Could not load model. Did you set your HF_TOKEN and accept the license on Hugging Face?")
        raise e
        
     # Ensure pad token is set (Llama 3 often lacks a default pad token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    
    # Explicitly update the model's generation config to suppress warnings
    model.generation_config.pad_token_id = tokenizer.pad_token_id
        
    return tokenizer, model

In [43]:
def create_prompt_str(tokenizer, question, options, context=None):
    """
    Creates the prompt string for Llama 3 using the chat template.
    Returns the string (not tensors) for batch tokenization.
    """
    options_text = "\n".join([f"{i+1}. {opt}" for i, opt in enumerate(options)])
    
    if context:
        system_msg = "You are a helpful assistant. Answer the multiple choice question based ONLY on the provided context."
        user_msg = (
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\n"
            f"Options:\n{options_text}\n\n"
            "Answer with the number of the correct option only (e.g., '1', '2', '3', or '4')."
        )
    else:
        system_msg = "You are a helpful assistant. Answer the multiple choice question based on your internal knowledge."
        user_msg = (
            f"Question: {question}\n\n"
            f"Options:\n{options_text}\n\n"
            "Answer with the number of the correct option only (e.g., '1', '2', '3', or '4')."
        )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    
    # Apply template but do NOT tokenize yet
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def get_model_response(model, tokenizer, messages):
    input_ids = tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=10,
        eos_token_id=terminators,
        do_sample=False, # Deterministic for evaluation
        temperature=0.0
    )
    
    response = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()

In [44]:
ds, unique_passages = load_data()
df = ds.to_pandas()
retriever = RAGRetriever(unique_passages)
tokenizer, model = load_llama()

Loading dataset facebook/belebele (eng_Latn)...
Found 488 unique passages to index.
Loading embedding model...
Encoding passages (this may take a moment)...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Index built with 488 documents.
Loading Llama-3.1-8B (4-bit quantized)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [45]:
def generate_batch(model, tokenizer, prompt_strs):
    inputs = tokenizer(
        prompt_strs, 
        return_tensors="pt", 
        padding=True, 
        truncation=True
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        eos_token_id=terminators,
        do_sample=False,
        temperature=None, 
        top_p=None
    )
    
    # Decode only the new tokens (slice outputs to remove input prompt)
    generated_tokens = outputs[:, inputs.input_ids.shape[1]:]
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

In [47]:
# 4. Run Experiment in Batches
BATCH_SIZE = 8 # Adjust based on your VRAM (8 is usually safe for 8GB+ w/ 4bit)

print("\n" + "="*50)
print(f"STARTING EXPERIMENT (Total Samples: {len(df)})")
print("="*50)

no_rag_preds = []
rag_preds = []

# Iterate through dataframe in chunks
for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Processing Batches"):
    # Get batch slice
    batch = df.iloc[i : i+BATCH_SIZE]
    
    # Extract fields
    questions = batch['question'].tolist()
    options_list = [
        [row['mc_answer1'], row['mc_answer2'], row['mc_answer3'], row['mc_answer4']] 
        for _, row in batch.iterrows()
    ]
    
    # --- Experiment A: NO RAG ---
    prompts_no_rag = []
    for q, opts in zip(questions, options_list):
        prompts_no_rag.append(create_prompt_str(tokenizer, q, opts, context=None))
        
    batch_preds_no_rag = generate_batch(model, tokenizer, prompts_no_rag)
    no_rag_preds.extend([p.strip() for p in batch_preds_no_rag])
    
    # --- Experiment B: WITH RAG ---
    # 1. Batch Retrieve (get top 1 doc for each question)
    retrieved_batch = retriever.retrieve(questions, k=1)
    contexts = [docs[0] for docs in retrieved_batch]
    
    # 2. Prepare Prompts
    prompts_rag = []
    for q, opts, ctx in zip(questions, options_list, contexts):
        prompts_rag.append(create_prompt_str(tokenizer, q, opts, context=ctx))
        
    # 3. Generate
    batch_preds_rag = generate_batch(model, tokenizer, prompts_rag)
    rag_preds.extend([p.strip() for p in batch_preds_rag])

# 5. Save Results
df['pred_no_rag'] = no_rag_preds
df['pred_rag'] = rag_preds

# Convert correct_answer_num (1-4) to string for comparison
df['correct_str'] = df['correct_answer_num'].astype(str)

# Calculate simple accuracy (check if prediction contains the correct number)
# Note: Llama 3 usually outputs just the digit "1", "2", etc. due to instructions
df['is_correct_no_rag'] = df.apply(lambda x: x['correct_str'] in x['pred_no_rag'], axis=1)
df['is_correct_rag'] = df.apply(lambda x: x['correct_str'] in x['pred_rag'], axis=1)

acc_no_rag = df['is_correct_no_rag'].mean()
acc_rag = df['is_correct_rag'].mean()

print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(f"Accuracy Without RAG: {acc_no_rag:.2%}")
print(f"Accuracy With RAG:    {acc_rag:.2%}")

# Show sample rows
print("\nSample Predictions:")
print(df[['question', 'correct_str', 'pred_no_rag', 'pred_rag']].head())

# Optional: Save to CSV
# df.to_csv("rag_benchmark_results.csv", index=False)


STARTING EXPERIMENT (Total Samples: 900)


Processing Batches:   0%|          | 0/113 [00:00<?, ?it/s]


RESULTS SUMMARY
Accuracy Without RAG: 41.44%
Accuracy With RAG:    75.78%

Sample Predictions:
                                            question correct_str pred_no_rag  \
0  According to the passage, what would not be co...           1           4   
1  When playing the accordion, which of the follo...           1           2   
2  Why do the images on television have their bor...           2           2   
3  According to the passage, which of the followi...           2           1   
4        Where was there a British garrison located?           3           2   

  pred_rag  
0        4  
1        1  
2        2  
3        2  
4        3  
